# Pretrained vs. random-init MobileNetV2

Compare a saved pretrained MobileNetV2 against a random-init baseline.

Assumes the dataset already exists in Google Drive as:
    BASE_DIR/train/<category>/*.jpg
    BASE_DIR/validation/<category>/*.jpg
    BASE_DIR/test/<category>/*.jpg
and that the saved model (mobilenetv2_pretrained.keras) is in BASE_DIR.

No data collection or pretrained-model training is done here. The only
training is the random-init baseline (use a GPU runtime).

**Before running:** set the runtime to GPU (Runtime > Change runtime type) and edit `BASE_DIR` in the first code cell.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tensorflow.keras import layers, models

## Settings - EDIT BASE_DIR to point at your copy/shortcut of the shared folder

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Marine_Classification"
MODEL_PATH = os.path.join(BASE_DIR, "mobilenetv2_pretrained.keras")

# Outputs go here. If you only have view access to the shared folder,
# change this to a folder you own, e.g. "/content/drive/MyDrive/my_results"
SAVE_DIR = os.path.join(BASE_DIR, "Model_Results")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 10  # keep equal to the original run for a fair comparison

## Mount Drive and check that everything we need is there

In [ ]:
drive.mount('/content/drive')

for path in [os.path.join(BASE_DIR, s) for s in ["train", "validation", "test"]]:
    if not os.path.isdir(path):
        raise FileNotFoundError(f"Missing dataset folder: {path}")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Saved model not found: {MODEL_PATH}")

os.makedirs(SAVE_DIR, exist_ok=True)

## Load the datasets (same settings as the original pipeline)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED
)

# Consistent class ordering across all splits
class_names = train_ds.class_names

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "validation"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=class_names,
    shuffle=False
)

# shuffle=False keeps the label order fixed so predictions line up
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=class_names,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

num_classes = len(class_names)
print("Classes:", class_names)
print("Number of classes:", num_classes)

## Load the saved pretrained model and score it on the test set

In [ ]:
mobilenet_model = tf.keras.models.load_model(MODEL_PATH)

test_loss, test_accuracy = mobilenet_model.evaluate(test_ds)
print("Pretrained test accuracy:", test_accuracy)
print("Pretrained test loss:", test_loss)

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred = mobilenet_model.predict(test_ds, verbose=0).argmax(axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

## Baseline B: same architecture, random (non-ImageNet) weights, trained from scratch with the same epochs, optimizer and data

In [ ]:
def build_model(weights, trainable_base):
    """Same architecture as the pretrained model; only initial weights differ."""
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=weights
    )
    base.trainable = trainable_base

    model_inputs = layers.Input(shape=(224, 224, 3))
    h = tf.keras.applications.mobilenet_v2.preprocess_input(model_inputs)
    h = base(h, training=trainable_base)
    h = layers.GlobalAveragePooling2D()(h)
    h = layers.Dropout(0.2)(h)
    model_outputs = layers.Dense(num_classes, activation="softmax")(h)

    model = models.Model(model_inputs, model_outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
tf.keras.utils.set_random_seed(SEED)
scratch_model = build_model(weights=None, trainable_base=True)

history_scratch = scratch_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

scratch_loss, scratch_accuracy = scratch_model.evaluate(test_ds)
print("Scratch test accuracy:", scratch_accuracy)
print("Scratch test loss:", scratch_loss)

y_pred_scratch = scratch_model.predict(test_ds, verbose=0).argmax(axis=1)

## Compare and save

In [ ]:
comparison_df = pd.DataFrame({
    "Pretrained MobileNetV2": {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    },
    "Random init (from scratch)": {
        "Accuracy": accuracy_score(y_true, y_pred_scratch),
        "Macro F1": f1_score(y_true, y_pred_scratch, average="macro",
                             zero_division=0)
    }
}).T

print("\nBASELINE COMPARISON (test set)")
display(comparison_df)
comparison_df.to_csv(os.path.join(SAVE_DIR, "baseline_comparison.csv"))

# Plot 1: Accuracy / macro-F1 side by side
comparison_df.plot.bar(figsize=(9, 5), ylim=(0, 1))
plt.title("Pretrained vs. Random-Init MobileNetV2 (Test Set)")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "baseline_comparison.png"),
            dpi=300, bbox_inches="tight")
plt.show()

# Plot 2: Scratch model's validation accuracy per epoch.
# (A loaded .keras file has no training history, so the pretrained curve
# is not available here.)
plt.figure(figsize=(9, 5))
plt.plot(history_scratch.history["val_accuracy"], label="Random init")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Random-Init Baseline: Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "baseline_val_accuracy.png"),
            dpi=300, bbox_inches="tight")
plt.show()

# Save the baseline model next to the original one (optional)
scratch_model.save(os.path.join(SAVE_DIR, "mobilenetv2_scratch.keras"))
print("Baseline model saved to:", SAVE_DIR)